In [2]:
!pip install insightface
!pip install onnxruntime-gpu  # or onnxruntime if you're on CPU
!pip install numpy opencv-python tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 9.0 MB/s eta 0:00:00ta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for insightface: filename=insightface-0.7.3-cp311-cp311-linux_x86_64.whl size=1057277 sha256=42f885c6718c29ae27f01de606ae811dfce3a8f82292e25e1312834c6039cd1e
  Stored in directory: /root/.cache/pip/wheels/27/d8/22/f52d858d16cd06e7b2e6aad34a1777dcfaf000be833bbf8146
Successfully built insightface
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 6.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.4 MB/s eta 0:00:00


In [6]:
from insightface.app import FaceAnalysis

face_detector = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider"])
face_detector.prepare(ctx_id=0, det_size=(640, 640))


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

In [8]:
import onnxruntime as ort
print("[DEBUG] ONNX Runtime device:", ort.get_device())  # Should return 'GPU'


[DEBUG] ONNX Runtime device: GPU


In [9]:
import onnxruntime as ort
print("[INFO] ONNX Runtime device in use:", ort.get_device())


[INFO] ONNX Runtime device in use: GPU


In [11]:
pip install av


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 51.0 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [17]:
# Deepfake Video Dataset Processing with InsightFace RetinaFace and GPU Batch Face Extraction

import os
import cv2
import numpy as np
from tqdm import tqdm
import shutil
import torch
import av
from insightface.app import FaceAnalysis

# --- Configuration ---
REAL_PATH = "/kaggle/input/faceforensics/FF++/real"   # Path to real videos
FAKE_PATH = "/kaggle/input/faceforensics/FF++/fake"   # Path to fake videos
OUTPUT_BASE = "extracted_faces"                        # Directory to store extracted faces
OUTPUT_FACE_SIZE = (200, 200)                          # Size to resize cropped face images
FRAME_COUNT = 15                                       # Number of frames to extract per video
MAX_VIDEOS = 200                                       # Limit number of videos to process
BATCH_SIZE = 8                                         # Number of frames to process in one batch

USE_GPU = torch.cuda.is_available()                    # Check if GPU is available
DEVICE_ID = 0 if USE_GPU else -1                       # Set device ID for InsightFace

# Initialize InsightFace RetinaFace model
'''face_detector = FaceAnalysis(
    name='buffalo_l',
    providers=['CUDAExecutionProvider'] if USE_GPU else ['CPUExecutionProvider']
)
face_detector.prepare(ctx_id=DEVICE_ID, det_size=(640, 640))'''

# Extract sample frames from a video using PyAV
# Selects FRAME_COUNT evenly spaced frames

def get_sample_frames_av(video_path: str, num_samples: int):
    container = av.open(video_path)
    stream = container.streams.video[0]
    total_frames = stream.frames
    if total_frames <= 0:
        return []
    frame_indices = np.linspace(0, total_frames - 1, num=num_samples, dtype=int)
    selected_frames = []

    for i, frame in enumerate(container.decode(video=0)):
        if i in frame_indices:
            img = frame.to_ndarray(format="bgr24")
            selected_frames.append(img)
        if i > frame_indices[-1]:
            break

    container.close()
    return selected_frames

# Extract faces from frames and save cropped results

def extract_and_save_faces(video_path: str, label: str):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    out_dir = os.path.join(OUTPUT_BASE, label, video_name)
    os.makedirs(out_dir, exist_ok=True)

    frames = get_sample_frames_av(video_path, FRAME_COUNT)
    batch = []
    indices = []

    for idx, frame in enumerate(tqdm(frames, desc=f"{label}/{video_name}", leave=False)):
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # Convert frame to RGB
        batch.append(rgb_frame)
        indices.append(idx)

        # Process batch when full or at last frame
        if len(batch) == BATCH_SIZE or idx == len(frames) - 1:
            try:
                # Run face detection
                results = face_detector.get(batch)  # List of face detections per image

                for b_idx, faces in enumerate(results):
                    if not isinstance(faces, list):
                        print(f"[WARN] Unexpected detection output format at index {b_idx}, skipping.")
                        continue
                    for f_idx, face in enumerate(faces):
                        cropped = face.crop_face()
                        if cropped is None or cropped.size == 0:
                            continue
                        if cropped.shape[0] == 0 or cropped.shape[1] == 0:
                            continue  # Skip invalid faces
                        try:
                            resized = cv2.resize(cropped, OUTPUT_FACE_SIZE)
                        except Exception as resize_err:
                            print(f"[WARN] Resize failed: {resize_err}")
                            continue
                        out_path = os.path.join(
                            out_dir,
                            f"frame_{indices[b_idx]:04d}_face_{f_idx:02d}.jpg"
                        )
                        cv2.imwrite(out_path, cv2.cvtColor(resized, cv2.COLOR_RGB2BGR))
            except Exception as e:
                print(f"[WARN] Skipping batch due to error: {e}")
            batch = []
            indices = []

# Process a directory of videos and extract faces

def process_videos(path: str, label: str):
    videos = [os.path.join(path, f) for f in os.listdir(path)
              if f.lower().endswith(('.mp4', '.mov', '.avi', '.mkv'))]
    videos = videos[:MAX_VIDEOS]  # Limit the number of videos

    for video in videos:
        extract_and_save_faces(video, label)

# Main execution function

def main():
    for lbl in ['real', 'fake']:
        os.makedirs(os.path.join(OUTPUT_BASE, lbl), exist_ok=True)

    process_videos(REAL_PATH, 'real')
    process_videos(FAKE_PATH, 'fake')

    # Zip output folder if running on Colab or Kaggle
    if os.getenv('KAGGLE_WORKING_DIR') or 'COLAB_GPU' in os.environ:
        shutil.make_archive(OUTPUT_BASE, 'zip', OUTPUT_BASE)
        print(f"Zipped extracted faces to {OUTPUT_BASE}.zip")

if __name__ == '__main__':
    main()


[WARN] Skipping batch due to error: 'list' object has no attribute 'shape'
[WARN] Skipping batch due to error: 'list' object has no attribute 'shape'


KeyboardInterrupt: 